# Day 13 Code And Outputs Walkthrough

This notebook explains the code and artifacts created for Day 13: NGBoost probability-quality evaluation. It is a companion to `03_ngboost_evaluation.ipynb`: that notebook is the report; this notebook is the implementation map and visual tour.

The focus is calibration and probabilistic usefulness, not point-prediction accuracy.

In [ ]:
from pathlib import Path
import ast
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

FIGURES = ROOT / "outputs" / "figures"

paths = {
    "evaluation_module": ROOT / "src" / "evaluation.py",
    "calibration_module": ROOT / "src" / "calibration.py",
    "runner": ROOT / "scripts" / "evaluate_ngboost.py",
    "evaluation_tests": ROOT / "tests" / "test_evaluation.py",
    "calibration_tests": ROOT / "tests" / "test_calibration.py",
    "ngboost_params": ROOT / "outputs" / "ngboost_distribution_params_v0.csv",
    "ngboost_bucket_probs": ROOT / "outputs" / "ngboost_bucket_probs_v0.csv",
    "day9_baseline": ROOT / "outputs" / "day9_empirical_baseline" / "empirical_baseline_predictions.csv",
    "coverage_report": ROOT / "outputs" / "coverage_report.csv",
    "residual_summary": ROOT / "outputs" / "standardized_residual_summary.csv",
    "bucket_brier": ROOT / "outputs" / "bucket_brier_scores.csv",
    "calibration_tables": ROOT / "outputs" / "calibration_tables.csv",
    "coverage_by_group": ROOT / "outputs" / "coverage_by_group.csv",
    "model_comparison": ROOT / "outputs" / "ngboost_evaluation_report.csv",
    "pit_histogram": FIGURES / "pit_histogram.png",
    "coverage_by_hour": FIGURES / "coverage_by_hour.png",
    "coverage_by_season": FIGURES / "coverage_by_season.png",
}

paths

## What Was Added Today

Day 13 added a reusable evaluation layer, calibration/plot helpers, a repeatable evaluation runner, tests, and generated outputs. The runner reads the real Day 11 NGBoost artifacts and Day 9 empirical baseline artifacts instead of fabricating input paths.

In [ ]:
inventory = []
for name, path in paths.items():
    inventory.append(
        {
            "artifact": name,
            "path": str(path.relative_to(ROOT)),
            "exists": path.exists(),
            "size_kb": round(path.stat().st_size / 1024, 1) if path.exists() else np.nan,
        }
    )
pd.DataFrame(inventory)

## Code Architecture

- `src/evaluation.py` contains metric and validation functions: NLL, coverage, PIT, standardized residuals, Brier, interval log loss, and probability validation.
- `src/calibration.py` contains calibration tables and matplotlib plotting helpers.
- `scripts/evaluate_ngboost.py` is the repeatable runner that loads artifacts, enforces chronological out-of-sample evaluation, aligns the empirical baseline, and writes reports/figures.
- `tests/test_evaluation.py` and `tests/test_calibration.py` lock down basic metric behavior and validation failures.

In [ ]:
def list_functions(path: Path) -> pd.DataFrame:
    tree = ast.parse(path.read_text(encoding="utf-8"))
    rows = []
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and not node.name.startswith("_"):
            rows.append(
                {
                    "file": str(path.relative_to(ROOT)),
                    "function": node.name,
                    "line": node.lineno,
                    "public": True,
                }
            )
    return pd.DataFrame(rows)

pd.concat(
    [
        list_functions(paths["evaluation_module"]),
        list_functions(paths["calibration_module"]),
        list_functions(paths["runner"]),
    ],
    ignore_index=True,
)

## Data Flow

```text
outputs/ngboost_distribution_params_v0.csv
        +
outputs/ngboost_bucket_probs_v0.csv
        +
outputs/day9_empirical_baseline/empirical_baseline_predictions.csv
        |
        v
scripts/evaluate_ngboost.py
        |
        +--> outputs/coverage_report.csv
        +--> outputs/standardized_residual_summary.csv
        +--> outputs/bucket_brier_scores.csv
        +--> outputs/calibration_tables.csv
        +--> outputs/coverage_by_group.csv
        +--> outputs/ngboost_evaluation_report.csv
        +--> outputs/figures/*.png
```

Two bucket schemas are intentionally kept separate:

- NGBoost market-bucket diagnostics use the saved moving Kalshi-style bucket positions: `market_bucket_0` through `market_bucket_5`.
- Empirical-baseline comparison uses Day 9 fixed forecast-error intervals: `(-inf, -3]`, `(-3, -1]`, `(-1, 1]`, `(1, 3]`, `(3, inf)`.

## Load Today Outputs

In [ ]:
params = pd.read_csv(paths["ngboost_params"])
coverage = pd.read_csv(paths["coverage_report"])
residuals = pd.read_csv(paths["residual_summary"])
bucket_brier = pd.read_csv(paths["bucket_brier"])
calibration_tables = pd.read_csv(paths["calibration_tables"])
coverage_by_group = pd.read_csv(paths["coverage_by_group"])
comparison = pd.read_csv(paths["model_comparison"])

summary = pd.DataFrame(
    [
        {"table": "ngboost params", "rows": len(params), "columns": len(params.columns)},
        {"table": "coverage report", "rows": len(coverage), "columns": len(coverage.columns)},
        {"table": "residual summary", "rows": len(residuals), "columns": len(residuals.columns)},
        {"table": "bucket brier", "rows": len(bucket_brier), "columns": len(bucket_brier.columns)},
        {"table": "calibration tables", "rows": len(calibration_tables), "columns": len(calibration_tables.columns)},
        {"table": "coverage by group", "rows": len(coverage_by_group), "columns": len(coverage_by_group.columns)},
        {"table": "model comparison", "rows": len(comparison), "columns": len(comparison.columns)},
    ]
)
summary

## Model Comparison Visuals

The empirical baseline is only compared on metrics it actually supports: fixed forecast-error interval log loss and Brier. NGBoost also has continuous Normal NLL, coverage, and residual metrics.

In [ ]:
display(comparison)

test_comparison = comparison[comparison["split"] == "test"].copy()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(test_comparison["model"], test_comparison["interval_log_loss"], color=["#4c78a8", "#f58518"])
axes[0].set_title("Test Interval Log Loss")
axes[0].set_ylabel("lower is better")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(test_comparison["model"], test_comparison["mean_bucket_brier"], color=["#4c78a8", "#f58518"])
axes[1].set_title("Test Mean Bucket Brier")
axes[1].set_ylabel("lower is better")
axes[1].tick_params(axis="x", rotation=20)

fig.tight_layout()
plt.show()

## Coverage Visuals

Coverage below the expected line means overconfidence. Coverage above the expected line means underconfidence.

In [ ]:
display(coverage)

plot_df = coverage[coverage["split"].isin(["validation", "test", "combined_out_of_sample"])].copy()
plot_df["level_label"] = (plot_df["level"] * 100).astype(int).astype(str) + "%"

fig, ax = plt.subplots(figsize=(9, 4.5))
for split, group in plot_df.groupby("split"):
    ax.plot(group["level_label"], group["actual_coverage"], marker="o", label=split)
ax.plot(plot_df.drop_duplicates("level_label")["level_label"], plot_df.drop_duplicates("level_label")["expected_coverage"], linestyle="--", color="black", label="expected")
ax.set_title("Actual Prediction Interval Coverage")
ax.set_xlabel("central interval")
ax.set_ylabel("coverage")
ax.set_ylim(0.35, 1.0)
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

## PIT Histogram

PIT values should look roughly uniform for a well-calibrated continuous predictive distribution. A U-shape suggests the distribution is too narrow; a hump suggests it is too wide; skew suggests directional bias.

In [ ]:
display(Image(filename=str(paths["pit_histogram"])))

## Standardized Residual Visuals

If residual std is greater than 1, predicted sigma is likely too small. If residual std is less than 1, predicted sigma is likely too large.

In [ ]:
display(residuals)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(residuals["split"], residuals["mean"], color="#72b7b2")
axes[0].axhline(0.0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Standardized Residual Mean")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(residuals["split"], residuals["std"], color="#e45756")
axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Standardized Residual Std")
axes[1].tick_params(axis="x", rotation=20)

fig.tight_layout()
plt.show()

## Market Bucket Brier And Calibration Gaps

`market_bucket_0` is the lower-tail market bucket position, and `market_bucket_5` is the upper-tail position. The labels are positions because final-temperature bucket names move with `forecast_high`.

In [ ]:
display(bucket_brier)

combined_brier = bucket_brier[bucket_brier["split"] == "combined_out_of_sample"].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(combined_brier["bucket"], combined_brier["brier_score"], color="#4c78a8")
axes[0].set_title("Combined OOS Brier By Market Bucket")
axes[0].set_ylabel("Brier score")
axes[0].tick_params(axis="x", rotation=30)

colors = np.where(combined_brier["calibration_gap"] >= 0, "#59a14f", "#e15759")
axes[1].bar(combined_brier["bucket"], combined_brier["calibration_gap"], color=colors)
axes[1].axhline(0.0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Mean Predicted Probability - Empirical Frequency")
axes[1].set_ylabel("calibration gap")
axes[1].tick_params(axis="x", rotation=30)

fig.tight_layout()
plt.show()

## Calibration Curves Created Today

In [ ]:
display(calibration_tables.head(20))

for image_path in sorted(FIGURES.glob("calibration_bucket_*.png")):
    print(image_path.name)
    display(Image(filename=str(image_path)))

## Coverage By Hour And Season

The runner keeps small groups and flags them with `enough_sample` rather than silently dropping them.

In [ ]:
display(coverage_by_group.head(40))
display(Image(filename=str(paths["coverage_by_hour"])))
if paths["coverage_by_season"].exists():
    display(Image(filename=str(paths["coverage_by_season"])))

## Output Checklist

These are the main Day 13 outputs and what each one is for.

In [ ]:
output_purpose = pd.DataFrame(
    [
        {"output": "outputs/ngboost_evaluation_report.csv", "purpose": "NGBoost vs empirical baseline on same forecast-error intervals"},
        {"output": "outputs/coverage_report.csv", "purpose": "50/80/90% central interval coverage by split"},
        {"output": "outputs/standardized_residual_summary.csv", "purpose": "Bias and sigma-quality diagnostics"},
        {"output": "outputs/bucket_brier_scores.csv", "purpose": "Market-bucket Brier, calibration gaps, and log loss"},
        {"output": "outputs/calibration_tables.csv", "purpose": "Binned calibration tables for selected market buckets and Day 9 comparison intervals"},
        {"output": "outputs/coverage_by_group.csv", "purpose": "80% coverage by hour, season, and forecast horizon"},
        {"output": "outputs/figures/pit_histogram.png", "purpose": "PIT diagnostic"},
        {"output": "outputs/figures/coverage_by_hour.png", "purpose": "Hour-level coverage diagnostic"},
        {"output": "outputs/figures/coverage_by_season.png", "purpose": "Season-level coverage diagnostic"},
    ]
)
output_purpose["exists"] = output_purpose["output"].map(lambda p: (ROOT / p).exists())
output_purpose

## Reproduce Day 13

Run this from the repository root to regenerate the outputs:

```powershell
python scripts\evaluate_ngboost.py
pytest
```

The runner explicitly guards against evaluating train rows, random splits, mismatched NGBoost/baseline rows, invalid probability sums, missing realized labels, and mixing the moving market-bucket schema with the Day 9 fixed forecast-error interval schema.

## Takeaway

Today's code says NGBoost has a real probability signal: it beats the Day 9 empirical baseline on the same test rows and same forecast-error intervals. But the calibration diagnostics are not clean enough to call it the unchecked primary probability signal. Test 80% coverage is below expected coverage, and standardized residual std is above 1, so the model is overconfident out of sample and likely underestimates sigma.